# 🚀 QuantDataPipeline — 量化數據中台

**全自動一鍵執行**：輸入參數 → 按下播放鍵 → 自動下載 + Greeks 計算 + 多週期特徵聚合

---

### 📋 使用說明
1. 在下方表單填入 **FinMind API Token**（免費註冊即可取得，可加速 API 限速）
2. 設定 **回溯天數**（從今天往回推幾天）
3. 勾選是否 **同步到 Google Drive**
4. 按下左側 ▶️ 播放鍵即可

> ⚠️ 首次執行會自動安裝依賴套件（約 30 秒），後續執行會直接跳過。

In [ ]:
#@title 🎛️ QuantDataPipeline 控制面板 { run: "auto", display-mode: "form" }
#@markdown ---
#@markdown ### 🔑 API 設定
FINMIND_API_TOKEN = '' #@param {type:"string"}
#@markdown > 免費註冊: [finmindtrade.com](https://finmindtrade.com/) — 有 Token 可將速率限制從 12s 降到 6s
#@markdown ---
#@markdown ### ⚙️ 管線參數
LOOKBACK_DAYS = 30 #@param {type:"slider", min:1, max:365, step:1}
SKIP_GREEKS = False #@param {type:"boolean"}
#@markdown > 勾選 `SKIP_GREEKS` 可僅下載原始資料，不計算 Greeks
#@markdown ---
#@markdown ### 💾 儲存設定
SYNC_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_PATH = '/content/drive/MyDrive/QuantData' #@param {type:"string"}
#@markdown ---

# ═══════════════════════════════════════════════════════════════
# 以下為自動執行邏輯，不需修改
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, os, time, shutil, json
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, HTML, clear_output

# ── 鎖定輸出高度 + 自動捲軸 ──
display(HTML("""
<style>
  .output_scroll { height: 420px !important; overflow-y: auto !important; }
  .output_wrapper { max-height: 420px !important; overflow-y: auto !important; }
  .output_area pre { font-family: 'Fira Code', 'Consolas', monospace; font-size: 13px; line-height: 1.6; }
</style>
<script>
  // 強制鎖定輸出區高度
  (function() {
    var output = document.querySelector('.output_scroll, .output_wrapper');
    if (output) {
      output.style.maxHeight = '420px';
      output.style.overflowY = 'auto';
    }
    // 自動捲到底
    var observer = new MutationObserver(function() {
      var el = document.querySelector('.output_scroll, .output_wrapper');
      if (el) el.scrollTop = el.scrollHeight;
    });
    var target = document.querySelector('.output_area');
    if (target) observer.observe(target, {childList: true, subtree: true});
  })();
</script>
"""))

# ── 列印工具 ──
def log(icon, msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"{icon} {ts} | {msg}", flush=True)

def header(title):
    print(f"\n{'━'*55}", flush=True)
    print(f"  {title}", flush=True)
    print(f"{'━'*55}", flush=True)

header('🚀 QuantDataPipeline 啟動中')
log('📅', f'回溯天數: {LOOKBACK_DAYS} | Greeks: {"開" if not SKIP_GREEKS else "關"} | Drive 同步: {"開" if SYNC_TO_DRIVE else "關"}')

# ═══════════════════════════════════════════════════════════════
# Phase 0: 環境準備
# ═══════════════════════════════════════════════════════════════
header('📦 Phase 0: 環境準備')

REPO_URL = 'https://github.com/hsp1234-web/SP_OP_20260220.git'
BRANCH = '3'
PROJECT_DIR = Path('/content/QuantDataPipeline')
REPO_DIR = Path('/content/SP_OP_20260220')

# Google Drive 掛載
if SYNC_TO_DRIVE:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        log('✅', 'Google Drive 已掛載')
    except Exception as e:
        log('⚠️', f'Drive 掛載失敗: {e} — 將以本地模式繼續')
        SYNC_TO_DRIVE = False

# Clone / Pull 最新代碼
if REPO_DIR.exists():
    log('🔄', '更新既有代碼...')
    subprocess.run(['git', 'pull'], cwd=str(REPO_DIR), capture_output=True)
else:
    log('📥', '首次下載代碼...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], capture_output=True)

PROJECT_DIR = REPO_DIR / 'QuantDataPipeline'
sys.path.insert(0, str(PROJECT_DIR))

# 安裝依賴
log('📦', '檢查 / 安裝依賴套件...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'polars', 'numba', 'scipy', 'numpy', 'requests', 'python-dotenv'],
    capture_output=True, text=True
)
log('✅', '依賴套件就緒')

# 寫入 .env
env_path = PROJECT_DIR / '.env'
env_path.write_text(f'FINMIND_API_TOKEN={FINMIND_API_TOKEN}\n')
log('🔑', f'API Token: {"已設定" if FINMIND_API_TOKEN else "未設定 (匿名模式, 12s/req)"}')

# ═══════════════════════════════════════════════════════════════
# Phase 1: 初始化管線
# ═══════════════════════════════════════════════════════════════
header('⚡ Phase 1: 資料下載')

os.chdir(str(PROJECT_DIR))

# 動態匯入（避免頂層 import 在安裝前失敗）
from core.config import DATA_DIR, DB_PATH
from core.db_metadata_manager import DBManager
from core.fetch_orchestrator import process_task, seed_tasks_from_dates
from fetchers.datasets.technical import trading_date
from fetchers.infrastructure.http_session import get_session

# 本地資料目錄
local_data = Path('/content/local_data')
local_data.mkdir(exist_ok=True)

# Drive 同步用
drive_data = Path(DRIVE_PATH) if SYNC_TO_DRIVE else None

# 如果 Drive 有既有 DB，複製到本地
if SYNC_TO_DRIVE and drive_data:
    drive_db = drive_data / 'status.db'
    if drive_db.exists():
        shutil.copy2(drive_db, DB_PATH)
        log('📋', '已從 Drive 還原 status.db')

# 初始化 DB
DBManager._reset_instance()
db = DBManager(DB_PATH)
session = get_session()

# 日期範圍
today = datetime.now()
end_date = today.strftime('%Y-%m-%d')
start_date = (today - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
log('📅', f'日期範圍: {start_date} → {end_date}')

# 取得交易日
try:
    trading_dates_df = trading_date.fetch_trading_dates(session, start_date, end_date)
    if trading_dates_df is None or trading_dates_df.is_empty():
        log('⚠️', '此範圍內無交易日')
        trading_dates_df = None
    else:
        n_dates = len(trading_dates_df)
        log('✅', f'找到 {n_dates} 個交易日')
except Exception as e:
    log('❌', f'取得交易日失敗: {e}')
    trading_dates_df = None

# 註冊 & 下載
TARGET_DATASETS = [('TaiwanOptionTick', 'TXO'), ('TaiwanFuturesTick', 'TX')]
COOLDOWN = 300

if trading_dates_df is not None:
    seed_tasks_from_dates(db, trading_dates_df, TARGET_DATASETS)
    pending = db.get_pending_tasks()
    total = len(pending)
    log('📊', f'待下載任務: {total} 個')

    done = 0
    for i, (task_id, tdate, dset, did) in enumerate(pending, 1):
        try:
            process_task(task_id, tdate, dset, did)
            status = db.get_task_status(task_id)
            if status == 3:
                log('➖', f'{tdate} {did:>3} | 無資料跳過')
            elif status == 1:
                log('✅', f'{tdate} {did:>3} | 下載成功')
                done += 1
            else:
                log('⏳', f'{tdate} {did:>3} | 待重試')
        except Exception as e:
            err = str(e).lower()
            if any(k in err for k in ['429', 'rate limit', 'quota']):
                log('🧊', f'API 額度耗盡 — 冷卻 {COOLDOWN}s...')
                time.sleep(COOLDOWN)
                try:
                    process_task(task_id, tdate, dset, did)
                    log('✅', f'{tdate} {did:>3} | 重試成功')
                    done += 1
                except:
                    log('❌', f'{tdate} {did:>3} | 重試失敗')
            else:
                log('❌', f'{tdate} {did:>3} | {str(e)[:40]}')

    log('📊', f'Phase 1 完成: {done}/{total} 個任務成功')

# ═══════════════════════════════════════════════════════════════
# Phase 2: Greeks 計算
# ═══════════════════════════════════════════════════════════════
if not SKIP_GREEKS:
    header('🧮 Phase 2: Greeks 特徵計算')

    from compute_greeks_pipeline import compute_greeks_for_date

    l1_tasks = db.get_tasks_by_status(1)
    opt_dates = set()
    for tid, tdate, dset, did in l1_tasks:
        if dset == 'TaiwanOptionTick':
            year = tdate.split('-')[0]
            gpath = DATA_DIR / year / 'GreeksFeatures' / f'TXO_Greeks_{tdate}.parquet'
            if not gpath.exists():
                fut_tid = f'{tdate}_TaiwanFuturesTick_TX'
                fut_st = db.get_task_status(fut_tid)
                if fut_st and fut_st >= 1:
                    opt_dates.add(tdate)

    computable = sorted(opt_dates)
    total_c = len(computable)
    log('📊', f'待計算日期: {total_c} 個')

    done_c = 0
    for i, d in enumerate(computable, 1):
        try:
            df, opath, ok = compute_greeks_for_date(d)
            if ok:
                db.update_task_status(f'{d}_TaiwanOptionTick_TXO', 2)
                log('✅', f'{d} | Greeks 完成 ({len(df):,} 筆)')
                done_c += 1
            else:
                log('➖', f'{d} | 無有效資料')
        except Exception as e:
            log('❌', f'{d} | {str(e)[:50]}')

    log('📊', f'Phase 2 完成: {done_c}/{total_c} 個日期成功')

# ═══════════════════════════════════════════════════════════════
# Phase 3: Drive 同步
# ═══════════════════════════════════════════════════════════════
if SYNC_TO_DRIVE and drive_data:
    header('☁️ Phase 3: Google Drive 同步')

    drive_data.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DB_PATH, drive_data / 'status.db')
    log('✅', 'status.db 已同步')

    # 同步 Parquet
    synced = 0
    if DATA_DIR.exists():
        for pq in DATA_DIR.rglob('*.parquet'):
            rel = pq.relative_to(DATA_DIR)
            dest = drive_data / 'data' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or dest.stat().st_size != pq.stat().st_size:
                shutil.copy2(pq, dest)
                synced += 1
    log('✅', f'已同步 {synced} 個 Parquet 檔案到 Drive')

# ═══════════════════════════════════════════════════════════════
# 完成統計
# ═══════════════════════════════════════════════════════════════
header('🏁 執行完畢')

stats = {
    '待處理': len(db.get_tasks_by_status(0)),
    'L1完成': len(db.get_tasks_by_status(1)),
    'L2完成': len(db.get_tasks_by_status(2)),
    '已跳過': len(db.get_tasks_by_status(3)),
}
total_all = sum(stats.values())

log('📊', f'任務總數: {total_all}')
for k, v in stats.items():
    bar = '█' * int(v / max(total_all, 1) * 20)
    log('  ', f'{k}: {v:4d} {bar}')

elapsed = time.time()
log('🎉', '管線執行完畢！')
